In [10]:
import os
import glob
import gc
import pandas as pd

folder_path = r'C:\8ssible-Healing-Seoul-Analysis\branch_SG\8SSIBLE_SG_DATA\S-DoT_NATURE_2023년(2023.01.01~2024.01.01)'

file_list = sorted(glob.glob(os.path.join(folder_path, 'S-DoT_NATURE_*.csv')))

use_cols = [
    '측정시간',
    '자치구',
    '지역',
    '소음 평균(dB)',
    '진동(x) 평균(mm/s)',
    '진동(y) 평균(mm/s)',
    '진동(z) 평균(mm/s)'
]

value_cols = [
    '소음 평균(dB)',
    '진동(x) 평균(mm/s)',
    '진동(y) 평균(mm/s)',
    '진동(z) 평균(mm/s)'
]

noise_results = []
vibration_results = []

for file in file_list:
    for chunk in pd.read_csv(
        file,
        encoding='cp949',
        usecols=use_cols,
        chunksize=50000,
        low_memory=False
    ):
        chunk['측정시간'] = pd.to_datetime(
            chunk['측정시간'],
            format='%Y-%m-%d_%H:%M:%S',
            errors='coerce'
        )

        chunk = chunk.dropna(subset=['측정시간'])

        chunk = chunk.loc[
            ~chunk['자치구'].isin(['Seoul_Grand_Park', 'mobile_type'])
        ]

        chunk['month'] = (
            chunk['측정시간'].dt.year.astype(str)
            + '.'
            + chunk['측정시간'].dt.month.astype(str)
        )

        chunk = chunk.loc[chunk['month'] != '2024.1']

        for col in value_cols:
            chunk[col] = pd.to_numeric(chunk[col], errors='coerce')

        noise_temp = (
            chunk.groupby(['month', '자치구', '지역'])[['소음 평균(dB)']]
            .mean()
            .reset_index()
        )

        vibration_temp = (
            chunk.groupby(['month', '자치구'])[
                ['진동(x) 평균(mm/s)', '진동(y) 평균(mm/s)', '진동(z) 평균(mm/s)']
            ]
            .mean()
            .reset_index()
        )

        noise_results.append(noise_temp)
        vibration_results.append(vibration_temp)

        del chunk, noise_temp, vibration_temp
        gc.collect()

noise_table = (
    pd.concat(noise_results, ignore_index=True)
    .groupby(['month', '자치구', '지역'])[['소음 평균(dB)']]
    .mean()
    .reset_index()
)

vibration_table = (
    pd.concat(vibration_results, ignore_index=True)
    .groupby(['month', '자치구'])[
        ['진동(x) 평균(mm/s)', '진동(y) 평균(mm/s)', '진동(z) 평균(mm/s)']
    ]
    .mean()
    .reset_index()
)

month_order = [f'2023.{i}' for i in range(1, 13)]

noise_table['month'] = pd.Categorical(
    noise_table['month'],
    categories=month_order,
    ordered=True
)

vibration_table['month'] = pd.Categorical(
    vibration_table['month'],
    categories=month_order,
    ordered=True
)

noise_table = noise_table.sort_values(['자치구', '지역', 'month']).reset_index(drop=True)
vibration_table = vibration_table.sort_values(['자치구', 'month']).reset_index(drop=True)

print('소음 표')
print(noise_table)

print('\n진동 표')
print(vibration_table)

소음 표
        month         자치구               지역  소음 평균(dB)
0      2023.1   Dobong-gu  commercial_area  44.305301
1      2023.2   Dobong-gu  commercial_area  42.534611
2      2023.3   Dobong-gu  commercial_area  43.796548
3      2023.4   Dobong-gu  commercial_area  41.910966
4      2023.5   Dobong-gu  commercial_area  38.835394
...       ...         ...              ...        ...
1538   2023.8  Yongsan-gu  roads_and_parks  57.842788
1539   2023.9  Yongsan-gu  roads_and_parks  54.280314
1540  2023.10  Yongsan-gu  roads_and_parks  54.088691
1541  2023.11  Yongsan-gu  roads_and_parks  54.531015
1542  2023.12  Yongsan-gu  roads_and_parks  54.290704

[1543 rows x 4 columns]

진동 표
       month         자치구  진동(x) 평균(mm/s)  진동(y) 평균(mm/s)  진동(z) 평균(mm/s)
0     2023.1   Dobong-gu        0.169989        0.447757        6.724775
1     2023.2   Dobong-gu        0.024756        0.068489        1.032303
2     2023.3   Dobong-gu        0.024829        0.067972        1.027917
3     2023.4   Dobong-gu

In [12]:
print('소음 표')
print(noise_table)

print('\n진동 표')
print(vibration_table)

output_folder = r'C:\8ssible-Healing-Seoul-Analysis\branch_SG\8SSIBLE_SG_DATA'

noise_table.to_csv(
    os.path.join(output_folder, 'monthly_noise_table.csv'),
    index=False,
    encoding='cp949'
)

vibration_table.to_csv(
    os.path.join(output_folder, 'monthly_vibration_table.csv'),
    index=False,
    encoding='cp949'
)


소음 표
        month         자치구               지역  소음 평균(dB)
0      2023.1   Dobong-gu  commercial_area  44.305301
1      2023.2   Dobong-gu  commercial_area  42.534611
2      2023.3   Dobong-gu  commercial_area  43.796548
3      2023.4   Dobong-gu  commercial_area  41.910966
4      2023.5   Dobong-gu  commercial_area  38.835394
...       ...         ...              ...        ...
1538   2023.8  Yongsan-gu  roads_and_parks  57.842788
1539   2023.9  Yongsan-gu  roads_and_parks  54.280314
1540  2023.10  Yongsan-gu  roads_and_parks  54.088691
1541  2023.11  Yongsan-gu  roads_and_parks  54.531015
1542  2023.12  Yongsan-gu  roads_and_parks  54.290704

[1543 rows x 4 columns]

진동 표
       month         자치구  진동(x) 평균(mm/s)  진동(y) 평균(mm/s)  진동(z) 평균(mm/s)
0     2023.1   Dobong-gu        0.169989        0.447757        6.724775
1     2023.2   Dobong-gu        0.024756        0.068489        1.032303
2     2023.3   Dobong-gu        0.024829        0.067972        1.027917
3     2023.4   Dobong-gu